In [ ]:
import tensorflow as tf
tf.test.gpu_device_name()

''

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install tensorflow

In [ ]:
!pip install opencv-python

In [ ]:
import os
import gc
import numpy as np
import cv2
import glob
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Conv2D, MaxPool2D, Flatten
from keras import Input
from keras import utils

TRAIN_LABEL_DIRECTORY = "/content/gdrive/MyDrive/Disertatie/datasets/2000_images/train/labels"
TRAIN_IMAGE_DIRECTORY = "/content/gdrive/MyDrive/Disertatie/datasets/2000_images/train/images"
# TEST_LABEL_DIRECTORY = "/content/gdrive/MyDrive/Disertatie/datasets/2000_images/valid/labels"
# TEST_IMAGE_DIRECTORY = "/content/gdrive/MyDrive/Disertatie/datasets/2000_images/valid/images"
# TRAIN_LABEL_DIRECTORY = "//content//gdrive//MyDrive//Disertatie//datasets//5500_images//train//labels"
# TRAIN_IMAGE_DIRECTORY = "//content//gdrive//MyDrive//Disertatie//datasets//5500_images//train//images"
TEST_LABEL_DIRECTORY = "/content/gdrive/MyDrive/Disertatie/datasets/5500_images/valid/labels"
TEST_IMAGE_DIRECTORY = "/content/gdrive/MyDrive/Disertatie/datasets/5500_images/valid/images"

print(tf.__version__)

2.15.0


## Tests

In [ ]:
for filename in os.listdir(TRAIN_LABEL_DIRECTORY):
    if os.path.isfile(os.path.join(TRAIN_LABEL_DIRECTORY, filename)):
        with open(os.path.join(TRAIN_LABEL_DIRECTORY, filename), 'r') as file:
            print(f"Contents of {filename}:")
            print(file.read())

Contents of WEB05423.txt:


KeyboardInterrupt: 

### labels

In [ ]:
FILENAME = 'AoF04009.txt'

with open(os.path.join(TRAIN_LABEL_DIRECTORY, FILENAME), 'r') as file:
    print(f"Contents of {FILENAME}:")
    lines = file.readlines()
    print(lines[0].strip().split(' '))
    label = []
    for line in lines:
        label.append(line.strip().split(' ')[0])
    print(label)

Contents of AoF04009.txt:
['0', '0.494140625', '0.3104166666666667', '0.45234375000000004', '0.4708333333333333']
['0', '1', '1', '1', '1']


### images

In [ ]:
for images in glob.iglob(f'{TRAIN_IMAGE_DIRECTORY}/*'):
    if (images.endswith(".jpg")):
        print(images)

Streaming output truncated to the last 5000 lines.
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05906.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05851.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05928.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05909.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05892.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05903.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05873.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05857.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05882.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05878.jpg
/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/WEB05850.jpg
/content/gdrive/MyDrive/Disertatie/datasets/550

In [ ]:
img = cv2.imread("/content/gdrive/MyDrive/Disertatie/datasets/5500_images/train/images/AoF04009.jpg")
resized_image = cv2.resize(img, (640, 480), interpolation=cv2.INTER_LINEAR)
conv_img = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)
print(np.array(conv_img).shape)

(480, 640, 3)


## Prepare labels:
* 0 -> not detected
* 1 -> smoke detected
* 2 -> fire detected
* 3 -> fire and smoke detected

In [ ]:
def prepare_labels(DIRECTORY, filename):
    with open(os.path.join(DIRECTORY, filename), 'r') as file:
        lines = file.readlines()
        label = []
        for line in lines:
            label.append(line.strip().split(' ')[0])
        if ('0' in label) and not('1' in label):
            return 1
        elif not('0' in label) and ('1' in label):
            return 2
        elif ('0' in label) and ('1' in label):
            return 3
        elif not(('0' in label) and ('1' in label)) :
            return 0

### Train labels

In [ ]:
y = []
for filename in os.listdir(TRAIN_LABEL_DIRECTORY):
    if os.path.isfile(os.path.join(TRAIN_LABEL_DIRECTORY, filename)):
        y.append(prepare_labels(TRAIN_LABEL_DIRECTORY, filename))
y_train = np.array(y)
del y
gc.collect()

0

In [ ]:
y_train = y_train.reshape((-1,1))
gc.collect()
print(y_train.shape)

(2000, 1)


### Test labels

In [ ]:
y = []
for filename in os.listdir(TEST_LABEL_DIRECTORY):
    if os.path.isfile(os.path.join(TEST_LABEL_DIRECTORY, filename)):
        y.append(prepare_labels(TEST_LABEL_DIRECTORY, filename))
y_test = np.array(y)
del y
gc.collect()

0

In [ ]:
y_test = y_test.reshape((-1,1))
gc.collect()
print(y_test.shape)

(550, 1)


In [ ]:
hist_df = pd.DataFrame(y_test)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/y_test.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

## Prepare images

### Train images

In [ ]:
image_data = []
for image_file in glob.iglob(f'{TRAIN_IMAGE_DIRECTORY}/*'):
    if (image_file.endswith(".png") or image_file.endswith(".jpg") or image_file.endswith(".jpeg")):
        image = cv2.imread(image_file)
        resized_image = cv2.resize(image, (360, 240), interpolation=cv2.INTER_LINEAR)
        img = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)
        image_data.append(np.array(img))
X_train = np.array(image_data)
del image_data, image, resized_image, img
gc.collect()

0

In [ ]:
gc.collect()

0

In [ ]:
print(X_train[0].shape)

(240, 360, 3)


In [ ]:
X_train = X_train.reshape(X_train.shape[0], 360, 240, 3)
X_train = X_train.astype('float32')
gc.collect()

0

### Test images

In [ ]:
image_data = []
for image_file in glob.iglob(f'{TEST_IMAGE_DIRECTORY}/*'):
    if (image_file.endswith(".png") or image_file.endswith(".jpg") or image_file.endswith(".jpeg")):
        image = cv2.imread(image_file)
        resized_image = cv2.resize(image, (360, 240), interpolation=cv2.INTER_LINEAR)
        img = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)
        image_data.append(np.array(img))
X_test = np.array(image_data)
del image_data, image, resized_image, img
gc.collect()

0

In [ ]:
X_test = X_test.reshape(X_test.shape[0], 360, 240, 3)
X_test = X_test.astype('float32')
gc.collect()

0

## Normalization

In [ ]:
X_train /= 255
X_test /= 255

In [ ]:
n_classes = 4
print("Shape before one-hot encoding: ", y_train.shape)
Y_train = utils.to_categorical(y_train, n_classes)
Y_test = utils.to_categorical(y_test, n_classes)
print("Shape after one-hot encoding: ", Y_train.shape)

Shape before one-hot encoding:  (2000, 1)
Shape after one-hot encoding:  (2000, 4)


## Conv NN architecture

In [ ]:
model = Sequential()
# input
model.add(Input(shape=(360, 240, 3)))
# conv
model.add(Conv2D(64, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
model.add(Conv2D(128, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
model.add(Conv2D(128, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
# pooling
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.25))
# conv
model.add(Conv2D(256, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
model.add(Conv2D(256, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
# pooling
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.25))
# conv
model.add(Conv2D(512, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
model.add(Conv2D(512, kernel_size=(3,3), strides=(1,1), padding='same', activation='relu'))
# pooling
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.25))
# input from conv
model.add(Flatten())
# hidden layer's
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
# output layer
model.add(Dense(4, activation='softmax'))

# compile
model.compile(loss='categorical_crossentropy',
              metrics=['accuracy'],
              optimizer='adam')

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 360, 240, 64)      1792      
                                                                 
 conv2d_1 (Conv2D)           (None, 360, 240, 128)     73856     
                                                                 
 conv2d_2 (Conv2D)           (None, 360, 240, 128)     147584    
                                                                 
 max_pooling2d (MaxPooling2  (None, 180, 120, 128)     0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 180, 120, 128)     0         
                                                                 
 conv2d_3 (Conv2D)           (None, 180, 120, 256)     295168    
                                                        

### Training

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model1.weights.h5'

In [ ]:
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoints_save_path,
    save_weights_only=True,
    monitor='val_accuracy',
    verbose=1,
    mode='auto',
    save_best_only=False,
    save_freq="epoch")

In [ ]:
with tf.device('/device:GPU:0'):
  history = model.fit(X_train,
                      Y_train,
                      batch_size=16,
                      epochs=50,
                      validation_data=(X_test, Y_test),
                      verbose=1,
                      callbacks=[model_checkpoint_callback])

Epoch 1/50
125/125 [==============================] - ETA: 0s - loss: 1.1657 - accuracy: 0.5820
Epoch 1: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model1.weights.h5
125/125 [==============================] - 196s 1s/step - loss: 1.1657 - accuracy: 0.5820 - val_loss: 1.0852 - val_accuracy: 0.6550
Epoch 2/50
125/125 [==============================] - ETA: 0s - loss: 0.9734 - accuracy: 0.6355
Epoch 2: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model1.weights.h5
125/125 [==============================] - 153s 1s/step - loss: 0.9734 - accuracy: 0.6355 - val_loss: 0.9465 - val_accuracy: 0.7300
Epoch 3/50
125/125 [==============================] - ETA: 0s - loss: 0.9244 - accuracy: 0.6510
Epoch 3: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model1.weights.h5
125/125 [==============================] - 148s 1s/step - loss: 0.9244 - accuracy: 0.6510 - val_loss: 0.9718 - val_accuracy: 0.7200
Epoch 4/50
12

### Save training history

In [ ]:
import pandas as pd

hist_df = pd.DataFrame(history.history)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/history_model1.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

### load model

In [ ]:
model.load_weights(checkpoints_save_path)

### test model

In [ ]:
results = model.evaluate(X_test, Y_test, batch_size=16)
print("loss, accuracy:", results)

13/13 [==============================] - 21s 708ms/step - loss: 1.3601 - accuracy: 0.6200
loss, accuracy: [1.360090970993042, 0.6200000047683716]


### resume train

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model100.weights.h5'

In [ ]:
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoints_save_path,
    save_weights_only=True,
    monitor='val_accuracy',
    verbose=1,
    mode='auto',
    save_best_only=False,
    save_freq="epoch")

In [ ]:
with tf.device('/device:GPU:0'):
  history = model.fit(X_train, Y_train, batch_size=16, epochs=50, validation_data=(X_test, Y_test), verbose=1, callbacks=[model_checkpoint_callback])

Epoch 1/50
125/125 [==============================] - ETA: 0s - loss: 0.3827 - accuracy: 0.8485
Epoch 1: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5
125/125 [==============================] - 179s 1s/step - loss: 0.3827 - accuracy: 0.8485 - val_loss: 1.2750 - val_accuracy: 0.6350
Epoch 2/50
125/125 [==============================] - ETA: 0s - loss: 0.3899 - accuracy: 0.8510
Epoch 2: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5
125/125 [==============================] - 151s 1s/step - loss: 0.3899 - accuracy: 0.8510 - val_loss: 1.1741 - val_accuracy: 0.6300
Epoch 3/50
125/125 [==============================] - ETA: 0s - loss: 0.3586 - accuracy: 0.8650
Epoch 3: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5
125/125 [==============================] - 158s 1s/step - loss: 0.3586 - accuracy: 0.8650 - val_loss: 1.1778 - val_accuracy: 0.6200
Epoch 4

#### save training data

In [ ]:
import pandas as pd

hist_df = pd.DataFrame(history.history)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/history_model100.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

### load 100 epochs model

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5'

In [ ]:
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoints_save_path,
    save_weights_only=True,
    monitor='val_accuracy',
    verbose=1,
    mode='auto',
    save_best_only=False,
    save_freq="epoch")

In [ ]:
with tf.device('/device:GPU:0'):
  history = model.fit(X_train, Y_train, batch_size=16, epochs=50, validation_data=(X_test, Y_test), verbose=1, callbacks=[model_checkpoint_callback])

Epoch 1/50
125/125 [==============================] - ETA: 0s - loss: 0.1441 - accuracy: 0.9510
Epoch 1: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5
125/125 [==============================] - 191s 1s/step - loss: 0.1441 - accuracy: 0.9510 - val_loss: 1.8465 - val_accuracy: 0.6000
Epoch 2/50
125/125 [==============================] - ETA: 0s - loss: 0.1243 - accuracy: 0.9550
Epoch 2: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5
125/125 [==============================] - 166s 1s/step - loss: 0.1243 - accuracy: 0.9550 - val_loss: 2.1798 - val_accuracy: 0.6350
Epoch 3/50
125/125 [==============================] - ETA: 0s - loss: 0.1526 - accuracy: 0.9450
Epoch 3: saving model to /content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5
125/125 [==============================] - 160s 1s/step - loss: 0.1526 - accuracy: 0.9450 - val_loss: 1.7285 - val_accuracy: 0.6050
Epoch 4

In [ ]:
import pandas as pd

hist_df = pd.DataFrame(history.history)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/history_model150.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

## Testing models

### 50 epochs model

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model1.weights.h5'

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
results = model.evaluate(X_test, Y_test, batch_size=16)
print("loss, accuracy:", results)

35/35 [==============================] - 1313s 37s/step - loss: 1.5909 - accuracy: 0.6164
loss, accuracy: [1.5908983945846558, 0.6163636445999146]


### 100 epochs model

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model100.weights.h5'

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
results = model.evaluate(X_test, Y_test, batch_size=16)
print("loss, accuracy:", results)

35/35 [==============================] - 1312s 37s/step - loss: 3.3139 - accuracy: 0.5873
loss, accuracy: [3.3138740062713623, 0.5872727036476135]


### 150 epochs model

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5'

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
results = model.evaluate(X_test, Y_test, batch_size=16)
print("loss, accuracy:", results)

35/35 [==============================] - 1311s 37s/step - loss: 5.2158 - accuracy: 0.6018
loss, accuracy: [5.215788841247559, 0.6018182039260864]


#Inferenta pentru calculul histogramelor

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model1.weights.h5'

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
print(Y_test[0])

[0. 0. 1. 0.]


In [ ]:
print(X_test[0].shape)

(360, 240, 3)


In [ ]:
print(X_test[:0].shape)

(0, 360, 240, 3)


In [ ]:
prediction = model.predict(X_test)


18/18 [==============================] - 1291s 71s/step
[[8.2743112e-03 4.1765854e-02 4.5448518e-01 4.9547467e-01]
 [3.2397020e-01 1.6510697e-02 7.8945741e-02 5.8057332e-01]
 [7.2235911e-04 8.0060982e-04 3.3455040e-02 9.6502203e-01]
 ...
 [5.4040702e-05 3.0857460e-03 2.1064460e-01 7.8621566e-01]
 [8.0762923e-08 7.4992349e-06 2.1166373e-02 9.7882611e-01]
 [3.5100947e-03 3.9620031e-02 3.7748346e-01 5.7938641e-01]]


In [ ]:
import pandas as pd

hist_df = pd.DataFrame(prediction)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/inference_for_conf_mat.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

In [ ]:
for _ in prediction:
  print(_)

[0.00827431 0.04176585 0.45448518 0.49547467]
[0.3239702  0.0165107  0.07894574 0.5805733 ]
[7.223591e-04 8.006098e-04 3.345504e-02 9.650220e-01]
[2.5316156e-05 5.9666158e-06 1.4541778e-02 9.8542684e-01]
[2.9263017e-04 3.8570917e-05 2.9009614e-02 9.7065914e-01]
[6.3915201e-04 6.0245278e-04 4.2722806e-02 9.5603561e-01]
[1.4809331e-04 2.5446489e-05 2.5946604e-02 9.7387981e-01]
[0.00298499 0.04077954 0.45839113 0.49784437]
[0.03518286 0.12527716 0.12199069 0.7175494 ]
[0.00937258 0.00979953 0.16846454 0.8123633 ]
[0.3523102  0.07407297 0.10085877 0.47275805]
[0.00711443 0.00398767 0.1330062  0.85589164]
[0.04005606 0.00221968 0.02528248 0.9324417 ]
[6.0878944e-04 5.2911437e-06 9.8197861e-03 9.8956621e-01]
[0.02909744 0.02186133 0.18536213 0.76367915]
[2.2014932e-08 7.2946527e-08 3.4868449e-03 9.9651295e-01]
[7.59422898e-01 1.09454435e-04 5.59540757e-04 2.39908114e-01]
[0.07084216 0.0061461  0.13192682 0.7910849 ]
[0.01423423 0.01210213 0.13263687 0.8410268 ]
[0.28640112 0.05411435 0.06730

### 100 model inference

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model100.weights.h5'

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
prediction_100 = model.predict(X_test)

18/18 [==============================] - 1288s 71s/step


In [ ]:
hist_df = pd.DataFrame(prediction_100)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/inference_100_for_conf_mat.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

In [ ]:
for _ in prediction_100:
  print(_)

[0.00528643 0.00493992 0.3735321  0.6162416 ]
[0.25929472 0.00116776 0.0113311  0.72820634]
[3.6420113e-07 5.9564038e-09 1.4892531e-05 9.9998468e-01]
[1.04250056e-07 4.76978013e-10 2.33281572e-08 9.99999821e-01]
[1.1918045e-09 1.1570852e-10 1.8200288e-05 9.9998170e-01]
[1.0800540e-12 3.0040782e-14 1.3892013e-06 9.9999851e-01]
[5.2633234e-15 1.9198523e-15 5.5857690e-08 9.9999994e-01]
[1.4174005e-10 7.3712505e-04 9.9254191e-01 6.7209359e-03]
[1.5050098e-09 2.7067062e-09 1.8156040e-06 9.9999815e-01]
[2.6580537e-06 2.3232185e-06 2.2672121e-04 9.9976826e-01]
[1.6485833e-04 7.8818798e-03 8.7982154e-01 1.1213162e-01]
[7.706987e-10 4.973590e-12 8.597688e-07 9.999991e-01]
[3.5964531e-05 1.1160401e-09 1.6813114e-05 9.9994713e-01]
[1.7576952e-14 3.3938102e-17 4.5750674e-09 9.9999994e-01]
[0.00857932 0.00456711 0.20367512 0.7831783 ]
[1.850525e-11 2.048909e-13 5.361819e-06 9.999946e-01]
[9.9697077e-01 5.9616166e-12 8.5818618e-12 3.0292128e-03]
[2.9047582e-05 8.2597156e-09 8.1665399e-05 9.9988919e-

### 150 epochs inference

In [ ]:
checkpoints_save_path = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/model150.weights.h5'

In [ ]:
model.load_weights(checkpoints_save_path)

In [ ]:
prediction_150 = model.predict(X_test)

18/18 [==============================] - 1275s 71s/step


In [ ]:
hist_df = pd.DataFrame(prediction_150)

# or save to csv:
hist_csv_file = '/content/gdrive/MyDrive/Disertatie/Modele/CNN_checkpoints/inference_150_for_conf_mat.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

In [ ]:
for _ in prediction_150:
  print(_)

[0.00126669 0.10114306 0.738532   0.15905826]
[0.01536993 0.02711835 0.00902481 0.9484869 ]
[1.1589313e-09 1.4049030e-10 3.7957609e-08 9.9999994e-01]
[1.9188916e-13 2.0147375e-15 1.4548382e-12 9.9999994e-01]
[7.0107958e-10 6.7106043e-10 1.8572447e-09 9.9999994e-01]
[5.3983853e-15 6.5785620e-14 9.5959054e-09 9.9999994e-01]
[9.8186526e-20 4.2908217e-22 1.1963048e-11 9.9999994e-01]
[1.6659909e-15 2.3650653e-04 9.9938434e-01 3.7918641e-04]
[2.3302718e-03 5.3721829e-03 2.2670729e-04 9.9207085e-01]
[1.9439612e-07 1.5643782e-07 6.0140483e-06 9.9999362e-01]
[0.09467922 0.11275632 0.45951697 0.3330475 ]
[7.8465651e-11 7.0710018e-13 7.1681767e-09 9.9999994e-01]
[0.3959227  0.01841951 0.05843354 0.5272243 ]
[4.4907923e-23 6.9270492e-27 2.8827091e-16 9.9999994e-01]
[2.1129209e-07 4.2605127e-08 4.0431914e-05 9.9995929e-01]
[9.9565000e-22 1.1421450e-22 1.5075026e-10 9.9999994e-01]
[9.9997097e-01 9.9215605e-14 1.4263625e-14 2.8976026e-05]
[1.7223238e-15 1.0038214e-16 7.1126243e-09 9.9999994e-01]
[1.0